In [51]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION
# ============================================================

from pathlib import Path
from collections import defaultdict
from datetime import timedelta

import os
import json
import math
import time
import hashlib
import sqlite3
import re

import requests
import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import Point
from scipy.spatial import cKDTree


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = (
    Path.home()
    / "OneDrive"
    / "Desktop"
    / "family-toilet-search"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
)

GENERATED_DIR = (
    DATA_DIR
    / "generated"
)

RAW_DEMOGRAPHICS_DIR = (
    DATA_DIR
    / "raw_demographics"
)

RAW_SUPPORT_DIR = (
    DATA_DIR
    / "raw_support"
)


for folder in [
    GENERATED_DIR,
    RAW_SUPPORT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

MRT_EXITS_PATH = (
    PROJECT_DIR
    / "mrt_exits.geojson"
)

RESIDENTIAL_CLUSTERS_PATH = (
    GENERATED_DIR
    / "residential_clusters.json"
)

MRT_DESTINATIONS_PATH = (
    GENERATED_DIR
    / "mrt_destinations.json"
)

FAMILY_SUPPORT_LIBRARY_PATH = (
    GENERATED_DIR
    / "family_support_library.json"
)

PRELIMINARY_ROUTE_PATH = (
    GENERATED_DIR
    / "preliminary_residential_routes.json"
)

ROUTE_LIBRARY_PATH = (
    GENERATED_DIR
    / "residential_route_library.json"
)

ROUTE_PROGRESS_PATH = (
    GENERATED_DIR
    / "residential_route_build_progress.json"
)


# ============================================================
# RUNTIME CACHE
# ============================================================

RUNTIME_DIR = (
    Path.home()
    / ".young_family_access"
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CACHE_DB = (
    RUNTIME_DIR
    / "family_route_cache.sqlite"
)


# ============================================================
# SETTINGS
# ============================================================

EARTH_RADIUS_M = 6_371_000

MAX_RESIDENTIAL_CLUSTERS = 120

MIN_RESIDENTIAL_AREA_M2 = 20_000

ROUTE_CHECKPOINT_SPACING_M = 50

# How far from the walking line a support opportunity may be
# before we consider it relevant to the journey.
SUPPORT_ROUTE_DISTANCE_M = 80

# Support environments such as parks can be larger.
SUPPORT_ENVIRONMENT_DISTANCE_M = 100

MAPILLARY_MAX_DISTANCE_M = 100

MAPILLARY_CANDIDATES_PER_CHECKPOINT = 2


# ============================================================
# EXTERNAL SERVICES
# ============================================================

FOOT_ROUTER = (
    "https://routing.openstreetmap.de/"
    "routed-foot/route/v1/driving"
)

MAPILLARY_IMAGES_URL = (
    "https://graph.mapillary.com/images"
)


HTTP = requests.Session()

HTTP.headers.update({
    "User-Agent":
        "young-family-route-research/1.0"
})


print(
    "Project:",
    PROJECT_DIR
)

print(
    "Support-data folder:",
    RAW_SUPPORT_DIR
)

Project: C:\Users\MSI\OneDrive\Desktop\family-toilet-search
Support-data folder: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\raw_support


In [52]:
# ============================================================
# CELL 2 — GENERAL HELPERS
# ============================================================

def safe_json_dump(
    path,
    payload
):
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False
        )


def format_seconds(
    seconds
):
    return str(
        timedelta(
            seconds=max(
                0,
                int(seconds)
            )
        )
    )


def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2
):
    phi1 = math.radians(
        lat1
    )

    phi2 = math.radians(
        lat2
    )

    dphi = math.radians(
        lat2 - lat1
    )

    dlambda = math.radians(
        lon2 - lon1
    )

    a = (
        math.sin(
            dphi / 2
        ) ** 2
        +
        math.cos(phi1)
        *
        math.cos(phi2)
        *
        math.sin(
            dlambda / 2
        ) ** 2
    )

    return (
        EARTH_RADIUS_M
        *
        2
        *
        math.atan2(
            math.sqrt(a),
            math.sqrt(
                1 - a
            )
        )
    )


def sample_route(
    points,
    spacing_m=
        ROUTE_CHECKPOINT_SPACING_M
):
    if (
        not points
        or
        len(points) < 2
    ):
        return points.copy()

    sampled = [
        points[0].copy()
    ]

    accumulated = 0.0

    previous = (
        points[0]
    )

    for current in points[1:]:

        segment_distance = (
            haversine_m(
                previous["lat"],
                previous["lon"],
                current["lat"],
                current["lon"]
            )
        )

        accumulated += (
            segment_distance
        )

        if (
            accumulated
            >=
            spacing_m
        ):
            sampled.append(
                current.copy()
            )

            accumulated = 0.0

        previous = (
            current
        )

    last = (
        points[-1]
    )

    if (
        sampled[-1]["lat"]
        !=
        last["lat"]
        or
        sampled[-1]["lon"]
        !=
        last["lon"]
    ):
        sampled.append(
            last.copy()
        )

    return sampled


def clean_text(
    value
):
    if (
        value is None
        or
        pd.isna(value)
    ):
        return ""

    return (
        str(value)
        .strip()
        .upper()
    )


def slugify(
    value
):
    value = (
        str(value)
        .lower()
        .strip()
    )

    value = re.sub(
        r"[^a-z0-9]+",
        "-",
        value
    )

    return (
        value.strip("-")
    )


# ============================================================
# SQLITE CACHE
# ============================================================

with sqlite3.connect(
    CACHE_DB
) as connection:

    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS route_cache (
            cache_key TEXT PRIMARY KEY,
            payload_json TEXT NOT NULL
        )
        """
    )

    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS mapillary_cache (
            cache_key TEXT PRIMARY KEY,
            payload_json TEXT NOT NULL
        )
        """
    )

    connection.commit()


def cache_get(
    table,
    cache_key
):
    with sqlite3.connect(
        CACHE_DB
    ) as connection:

        row = connection.execute(
            f"""
            SELECT payload_json
            FROM {table}
            WHERE cache_key = ?
            """,
            (
                cache_key,
            )
        ).fetchone()

    if not row:
        return None

    return json.loads(
        row[0]
    )


def cache_set(
    table,
    cache_key,
    payload
):
    with sqlite3.connect(
        CACHE_DB
    ) as connection:

        connection.execute(
            f"""
            INSERT OR REPLACE
            INTO {table}
            (
                cache_key,
                payload_json
            )
            VALUES (?, ?)
            """,
            (
                cache_key,
                json.dumps(
                    payload
                )
            )
        )

        connection.commit()


print(
    "Helpers ready."
)

Helpers ready.


In [54]:
# ============================================================
# CELL 3 — DISCOVER LOCAL SUPPORT DATASETS
# ============================================================

support_files = []


for extension in [
    "*.geojson",
    "*.json",
    "*.gpkg",
    "*.shp"
]:
    support_files.extend(
        RAW_SUPPORT_DIR.glob(
            extension
        )
    )


support_files = sorted(
    list(
        dict.fromkeys(
            support_files
        )
    )
)


if not support_files:

    raise FileNotFoundError(
        "No support datasets found in "
        "data/raw_support/"
    )


print(
    "Local support datasets:"
)

for index, path in enumerate(
    support_files,
    start=1
):
    print(
        f"{index}.",
        path.name
    )

Local support datasets:
1. CentralNatureReserveAmenities.geojson
2. ParkConnectorLoop.geojson
3. ParkFacilities.geojson


In [55]:
# ============================================================
# CELL 4 — LOAD LOCAL SUPPORT DATASETS
# ============================================================

SUPPORT_DATASETS = {}


for path in support_files:

    try:

        gdf = gpd.read_file(
            path
        )


        # -----------------------------------------------
        # Force WGS84 for consistency
        # -----------------------------------------------

        if gdf.crs is not None:

            gdf = gdf.to_crs(
                epsg=4326
            )


        SUPPORT_DATASETS[
            path.name
        ] = gdf


        print()
        print(
            "=============================================="
        )

        print(
            path.name
        )

        print(
            "Rows:",
            len(gdf)
        )

        print(
            "Columns:",
            list(
                gdf.columns
            )
        )

        print(
            "Geometry:",
            gdf.geometry
            .geom_type
            .value_counts()
            .to_dict()
        )


    except Exception as error:

        print(
            "FAILED:",
            path.name,
            error
        )


print()
print(
    "Datasets successfully loaded:",
    len(
        SUPPORT_DATASETS
    )
)


CentralNatureReserveAmenities.geojson
Rows: 449
Columns: ['OBJECTID', 'BUILD_TYPE', 'BUILD_NAME', 'INC_CRC', 'FMEL_UPD_D', 'geometry']
Geometry: {'Point': 449}

ParkConnectorLoop.geojson
Rows: 878
Columns: ['OBJECTID', 'PARK', 'PCN_LOOP', 'MORE_INFO', 'INC_CRC', 'FMEL_UPD_D', 'SHAPE.LEN', 'geometry']
Geometry: {'LineString': 784, 'MultiLineString': 94}

ParkFacilities.geojson
Rows: 14754
Columns: ['OBJECTID', 'UNIQUEID', 'NAME', 'CLASS', 'ADDITIONAL_INFO', 'INC_CRC', 'FMEL_UPD_D', 'geometry']
Geometry: {'Point': 14754}

Datasets successfully loaded: 3


In [56]:
# ============================================================
# CELL 5 — SUPPORT DATA HELPERS
# ============================================================

def normalize_text(
    value
):

    if (
        value is None
        or
        pd.isna(
            value
        )
    ):

        return ""


    return (
        str(value)
        .strip()
        .upper()
    )


def row_text(
    row
):

    text_parts = []


    for column in row.index:

        if (
            column
            ==
            "geometry"
        ):

            continue


        value = row.get(
            column
        )


        text = normalize_text(
            value
        )


        if text:

            text_parts.append(
                text
            )


    return " | ".join(
        text_parts
    )


def first_existing_value(
    row,
    possible_columns
):

    for column in possible_columns:

        if (
            column
            in
            row.index
        ):

            value = row.get(
                column
            )


            if (
                value is not None
                and
                not pd.isna(
                    value
                )
                and
                str(value).strip()
            ):

                return str(
                    value
                ).strip()


    return None


print(
    "Support helpers ready."
)

Support helpers ready.


In [57]:
# ============================================================
# CELL 6 — FAMILY SUPPORT TYPOLOGY
# ============================================================

SUPPORT_KEYWORDS = {

    # --------------------------------------------------------
    # Actual stopping / sitting opportunities
    # --------------------------------------------------------

    "direct_rest": [

        "BENCH",
        "SEAT",
        "SEATING",
        "REST AREA",
        "REST POINT",
        "PICNIC TABLE",
        "PICNIC",
        "SHELTER",
        "PAVILION",
        "GAZEBO",
        "PERGOLA",
        "HUT"
    ],


    # --------------------------------------------------------
    # Caregiving infrastructure
    # --------------------------------------------------------

    "family_care": [

        "NURSING",
        "DIAPER",
        "NAPPY",
        "BABY",
        "CHANGING",
        "FAMILY ROOM",
        "FAMILY TOILET",
        "TOILET",
        "WASHROOM"
    ],


    # --------------------------------------------------------
    # Places where families can reasonably pause
    # --------------------------------------------------------

    "pause_destination": [

        "PLAYGROUND",
        "PLAY AREA",
        "PLAYFIELD",
        "PLAY SPACE",
        "FITNESS",
        "VISITOR CENTRE",
        "VISITOR CENTER",
        "PLAZA",
        "COMMUNITY",
        "EVENT SPACE",
        "FOOD",
        "CAFE",
        "KIOSK",
        "RESTAURANT"
    ],


    # --------------------------------------------------------
    # Climatic refuge / protected environment
    # --------------------------------------------------------

    "climate_refuge": [

        "SHELTER",
        "PAVILION",
        "GAZEBO",
        "COVERED",
        "SHADE",
        "PERGOLA",
        "HUT"
    ]
}


def classify_support(
    text
):

    text = normalize_text(
        text
    )


    categories = []


    for category, keywords in (
        SUPPORT_KEYWORDS.items()
    ):

        for keyword in keywords:

            if (
                keyword
                in
                text
            ):

                categories.append(
                    category
                )

                break


    return categories


print(
    "Family-support typology ready."
)

Family-support typology ready.


In [58]:
# ============================================================
# CELL 7 — BUILD FAMILY-SUPPORT CANDIDATES
# ============================================================

SUPPORT_NODES = []

SUPPORT_ENVIRONMENTS = []

SUPPORT_CORRIDORS = []


node_counter = 0

environment_counter = 0

corridor_counter = 0


for dataset_name, gdf in (
    SUPPORT_DATASETS.items()
):

    dataset_upper = (
        dataset_name.upper()
    )


    print()
    print(
        "Processing:",
        dataset_name
    )


    # ========================================================
    # PARK CONNECTOR LOOP
    # ========================================================

    if (
        "PARKCONNECTOR"
        in
        dataset_upper.replace(
            "_",
            ""
        )
        .replace(
            " ",
            ""
        )
    ):

        for _, row in (
            gdf.iterrows()
        ):

            geometry = (
                row.geometry
            )


            if (
                geometry is None
                or
                geometry.is_empty
            ):

                continue


            corridor_counter += 1


            SUPPORT_CORRIDORS.append({

                "id":
                    f"corridor-{corridor_counter:06d}",

                "type":
                    "park_connector",

                "category":
                    "mobility_support",

                "name":
                    first_existing_value(
                        row,
                        [
                            "NAME",
                            "Name",
                            "name",
                            "PCN_NAME",
                            "DESCRIPTION"
                        ]
                    ),

                "sourceDataset":
                    dataset_name,

                "geometry":
                    geometry
            })


        continue


    # ========================================================
    # PARK FACILITIES + NATURE RESERVE AMENITIES
    # ========================================================

    for _, row in (
        gdf.iterrows()
    ):

        geometry = (
            row.geometry
        )


        if (
            geometry is None
            or
            geometry.is_empty
        ):

            continue


        text = (
            row_text(
                row
            )
        )


        categories = (
            classify_support(
                text
            )
        )


        if not categories:

            continue


        name = (
            first_existing_value(
                row,
                [
                    "NAME",
                    "Name",
                    "name",
                    "FACILITY",
                    "CLASS",
                    "DESCRIPTION",
                    "DESC"
                ]
            )
        )


        # ----------------------------------------------------
        # POINT FACILITY
        # ----------------------------------------------------

        if (
            geometry.geom_type
            ==
            "Point"
        ):

            node_counter += 1


            SUPPORT_NODES.append({

                "id":
                    f"support-{node_counter:06d}",

                "lat":
                    float(
                        geometry.y
                    ),

                "lon":
                    float(
                        geometry.x
                    ),

                "categories":
                    categories,

                "primaryCategory":
                    categories[0],

                "name":
                    name,

                "sourceDataset":
                    dataset_name,

                "sourceText":
                    text
            })


        # ----------------------------------------------------
        # POLYGON / LINE FACILITY
        # ----------------------------------------------------

        else:

            environment_counter += 1


            SUPPORT_ENVIRONMENTS.append({

                "id":
                    f"environment-{environment_counter:06d}",

                "categories":
                    categories,

                "primaryCategory":
                    categories[0],

                "name":
                    name,

                "sourceDataset":
                    dataset_name,

                "sourceText":
                    text,

                "geometry":
                    geometry
            })


print()
print(
    "=============================================="
)

print(
    "Support nodes:",
    len(
        SUPPORT_NODES
    )
)

print(
    "Support environments:",
    len(
        SUPPORT_ENVIRONMENTS
    )
)

print(
    "Park-connector corridors:",
    len(
        SUPPORT_CORRIDORS
    )
)

print(
    "=============================================="
)


Processing: CentralNatureReserveAmenities.geojson

Processing: ParkConnectorLoop.geojson

Processing: ParkFacilities.geojson

Support nodes: 1161
Support environments: 0
Park-connector corridors: 878


In [59]:
# ============================================================
# CELL 8 — SUPPORT SUMMARY
# ============================================================

if SUPPORT_NODES:

    category_counts = (
        pd.Series([
            node[
                "primaryCategory"
            ]
            for node in
            SUPPORT_NODES
        ])
        .value_counts()
    )


    print(
        "POINT SUPPORT CATEGORIES"
    )

    print(
        category_counts
    )


    print()
    print(
        "SAMPLE SUPPORT POINTS"
    )


    for node in (
        SUPPORT_NODES[:20]
    ):

        print(
            node[
                "id"
            ],
            "|",
            node[
                "primaryCategory"
            ],
            "|",
            node[
                "name"
            ]
        )


else:

    print(
        "No point support opportunities identified."
    )


print()
print(
    "Support environments:",
    len(
        SUPPORT_ENVIRONMENTS
    )
)


print(
    "Mobility corridors:",
    len(
        SUPPORT_CORRIDORS
    )
)

POINT SUPPORT CATEGORIES
pause_destination    888
family_care          153
direct_rest          119
climate_refuge         1
Name: count, dtype: int64

SAMPLE SUPPORT POINTS
support-000001 | direct_rest | None
support-000002 | direct_rest | None
support-000003 | direct_rest | None
support-000004 | family_care | None
support-000005 | family_care | None
support-000006 | family_care | None
support-000007 | pause_destination | None
support-000008 | pause_destination | None
support-000009 | family_care | None
support-000010 | direct_rest | None
support-000011 | direct_rest | None
support-000012 | direct_rest | None
support-000013 | direct_rest | None
support-000014 | direct_rest | None
support-000015 | direct_rest | None
support-000016 | direct_rest | None
support-000017 | direct_rest | None
support-000018 | family_care | None
support-000019 | direct_rest | None
support-000020 | direct_rest | None

Support environments: 0
Mobility corridors: 878


In [60]:
# ============================================================
# CELL 9 — SAVE FAMILY SUPPORT LIBRARY
# ============================================================

FAMILY_SUPPORT_LIBRARY_PATH = (
    GENERATED_DIR
    / "family_support_library.json"
)


SUPPORT_BY_ID = {

    node[
        "id"
    ]:
        node

    for node in
    SUPPORT_NODES
}


FAMILY_SUPPORT_LIBRARY = {

    "version":
        2,

    "definition":
        (
            "Recorded facilities where a young family may "
            "reasonably stop, obtain care, seek shelter, "
            "or pause during an everyday walking journey."
        ),

    "categories": {

        "direct_rest":
            "Explicit seating, shelter or rest infrastructure.",

        "family_care":
            (
                "Infrastructure supporting caregiving needs "
                "such as toilets, nursing or changing."
            ),

        "pause_destination":
            (
                "Family-oriented destination where stopping "
                "or waiting can reasonably occur."
            ),

        "climate_refuge":
            (
                "Sheltered or protected environment capable "
                "of providing climatic respite."
            )
    },

    "nodeCount":
        len(
            SUPPORT_NODES
        ),

    "environmentCount":
        len(
            SUPPORT_ENVIRONMENTS
        ),

    "corridorCount":
        len(
            SUPPORT_CORRIDORS
        ),

    "nodes":
        SUPPORT_NODES,

    "nodesById":
        SUPPORT_BY_ID
}


safe_json_dump(
    FAMILY_SUPPORT_LIBRARY_PATH,
    FAMILY_SUPPORT_LIBRARY
)


print(
    "Saved:",
    FAMILY_SUPPORT_LIBRARY_PATH
)

Saved: C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\family_support_library.json


In [61]:
# ============================================================
# CELL 10 — FAMILY-SUPPORT SPATIAL INDEX
# ============================================================

if SUPPORT_NODES:

    SUPPORT_MEAN_LAT = (
        np.mean([
            node[
                "lat"
            ]
            for node in
            SUPPORT_NODES
        ])
    )


    SUPPORT_LAT_SCALE = (
        111_320
    )


    SUPPORT_LON_SCALE = (
        111_320
        *
        math.cos(
            math.radians(
                SUPPORT_MEAN_LAT
            )
        )
    )


    SUPPORT_XY = (
        np.array([
            [
                node[
                    "lon"
                ]
                *
                SUPPORT_LON_SCALE,

                node[
                    "lat"
                ]
                *
                SUPPORT_LAT_SCALE
            ]

            for node in
            SUPPORT_NODES
        ])
    )


    SUPPORT_TREE = (
        cKDTree(
            SUPPORT_XY
        )
    )


else:

    SUPPORT_TREE = None


def support_ids_near_route(
    route_points,
    max_distance_m=
        SUPPORT_ROUTE_DISTANCE_M
):

    if (
        SUPPORT_TREE is None
        or
        not route_points
    ):

        return []


    found_indices = set()


    for point in route_points:

        x = (
            point[
                "lon"
            ]
            *
            SUPPORT_LON_SCALE
        )


        y = (
            point[
                "lat"
            ]
            *
            SUPPORT_LAT_SCALE
        )


        indices = (
            SUPPORT_TREE
            .query_ball_point(
                [
                    x,
                    y
                ],
                r=
                    max_distance_m
            )
        )


        found_indices.update(
            indices
        )


    return [
        SUPPORT_NODES[
            index
        ][
            "id"
        ]

        for index in
        sorted(
            found_indices
        )
    ]


print(
    "Support KD-tree ready."
)

Support KD-tree ready.


In [62]:
# ============================================================
# CELL 11 — SUPPORT ENVIRONMENT + CORRIDOR GDF
# ============================================================

environment_rows = []


for item in (
    SUPPORT_ENVIRONMENTS
):

    environment_rows.append({

        "id":
            item[
                "id"
            ],

        "name":
            item[
                "name"
            ],

        "primaryCategory":
            item[
                "primaryCategory"
            ],

        "sourceDataset":
            item[
                "sourceDataset"
            ],

        "geometry":
            item[
                "geometry"
            ]
    })


if environment_rows:

    SUPPORT_ENVIRONMENT_GDF = (
        gpd.GeoDataFrame(
            environment_rows,
            geometry="geometry",
            crs="EPSG:4326"
        )
        .to_crs(
            epsg=3414
        )
    )


else:

    SUPPORT_ENVIRONMENT_GDF = (
        gpd.GeoDataFrame(
            columns=[
                "id",
                "geometry"
            ],
            geometry="geometry",
            crs="EPSG:3414"
        )
    )


corridor_rows = []


for item in (
    SUPPORT_CORRIDORS
):

    corridor_rows.append({

        "id":
            item[
                "id"
            ],

        "name":
            item[
                "name"
            ],

        "geometry":
            item[
                "geometry"
            ]
    })


if corridor_rows:

    SUPPORT_CORRIDOR_GDF = (
        gpd.GeoDataFrame(
            corridor_rows,
            geometry="geometry",
            crs="EPSG:4326"
        )
        .to_crs(
            epsg=3414
        )
    )


else:

    SUPPORT_CORRIDOR_GDF = (
        gpd.GeoDataFrame(
            columns=[
                "id",
                "geometry"
            ],
            geometry="geometry",
            crs="EPSG:3414"
        )
    )


print(
    "Support environments:",
    len(
        SUPPORT_ENVIRONMENT_GDF
    )
)

print(
    "Support corridors:",
    len(
        SUPPORT_CORRIDOR_GDF
    )
)

Support environments: 0
Support corridors: 878


In [63]:
# ============================================================
# CELL 12 — LOAD URA LAND USE
# ============================================================

landuse_candidates = []


for pattern in [

    "*landuse*.geojson",

    "*land_use*.geojson",

    "*a8c3546b26712e35021f3a681d0353ae*.geojson"

]:

    landuse_candidates.extend(
        RAW_DEMOGRAPHICS_DIR.glob(
            pattern
        )
    )


landuse_candidates = list(
    dict.fromkeys(
        landuse_candidates
    )
)


if not landuse_candidates:

    raise FileNotFoundError(
        "URA land-use GeoJSON not found."
    )


LANDUSE_PATH = (
    landuse_candidates[0]
)


landuse_gdf = (
    gpd.read_file(
        LANDUSE_PATH
    )
)


print(
    "Land-use file:",
    LANDUSE_PATH.name
)

print(
    "Features:",
    len(
        landuse_gdf
    )
)


if (
    "LU_DESC"
    not in
    landuse_gdf.columns
):

    raise RuntimeError(
        "LU_DESC column missing."
    )

Land-use file: mp2025_landuse.geojson
Features: 113394


In [64]:
# ============================================================
# CELL 13 — RESIDENTIAL ORIGINS
# ============================================================

residential_gdf = (
    landuse_gdf[
        landuse_gdf[
            "LU_DESC"
        ]
        .astype(str)
        .str.upper()
        .str.contains(
            "RESIDENTIAL",
            na=False
        )
    ]
    .copy()
)


residential_gdf = (
    residential_gdf
    .to_crs(
        epsg=3414
    )
)


residential_gdf[
    "area_m2"
] = (
    residential_gdf
    .geometry
    .area
)


residential_gdf = (
    residential_gdf[
        residential_gdf[
            "area_m2"
        ]
        >=
        MIN_RESIDENTIAL_AREA_M2
    ]
    .copy()
)


residential_gdf[
    "representative_point"
] = (
    residential_gdf
    .geometry
    .representative_point()
)


residential_points = (
    gpd.GeoDataFrame(

        residential_gdf.drop(
            columns=[
                "geometry"
            ]
        ),

        geometry=
            residential_gdf[
                "representative_point"
            ],

        crs=
            residential_gdf.crs
    )
    .to_crs(
        epsg=4326
    )
)


residential_points = (
    residential_points
    .sort_values(
        "area_m2",
        ascending=False
    )
    .head(
        MAX_RESIDENTIAL_CLUSTERS
    )
    .reset_index(
        drop=True
    )
)


clusters = []


for index, row in (
    residential_points.iterrows()
):

    point = (
        row.geometry
    )


    clusters.append({

        "id":
            f"residential-{index + 1:04d}",

        "name":
            f"Residential Cluster {index + 1:03d}",

        "lat":
            float(
                point.y
            ),

        "lon":
            float(
                point.x
            ),

        "area_m2":
            float(
                row[
                    "area_m2"
                ]
            ),

        "source":
            "URA Master Plan residential land use"
    })


safe_json_dump(
    RESIDENTIAL_CLUSTERS_PATH,
    clusters
)


print(
    "Residential origins:",
    len(
        clusters
    )
)

Residential origins: 120


In [66]:
# ============================================================
# MRT DESTINATIONS — ROBUST VERSION
# ============================================================

import json
import re
import html
import numpy as np


# ============================================================
# LOAD GEOJSON
# ============================================================

if not MRT_EXITS_PATH.exists():

    raise FileNotFoundError(
        f"MRT GeoJSON not found: {MRT_EXITS_PATH}"
    )


with open(
    MRT_EXITS_PATH,
    "r",
    encoding="utf-8"
) as f:

    mrt_geojson = json.load(f)


features = (
    mrt_geojson.get(
        "features",
        []
    )
)


print(
    "MRT GeoJSON features:",
    len(features)
)


# ============================================================
# HTML / DESCRIPTION PARSER
# ============================================================

def strip_html(
    value
):

    if value is None:
        return ""

    text = str(
        value
    )

    text = html.unescape(
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return (
        text.strip()
    )


def extract_station_from_html(
    value
):

    if value is None:

        return None


    raw = html.unescape(
        str(value)
    )


    patterns = [

        # OneMap / HTML table style
        r"STN_NAME\s*</[^>]+>\s*<[^>]+>\s*([^<]+)",

        r"STATION_NAME\s*</[^>]+>\s*<[^>]+>\s*([^<]+)",

        r"STATION\s*</[^>]+>\s*<[^>]+>\s*([^<]+)",

        # STN_NAME : value
        r"STN_NAME\s*[:=]\s*([^<;,]+)",

        r"STATION_NAME\s*[:=]\s*([^<;,]+)"
    ]


    for pattern in patterns:

        match = re.search(
            pattern,
            raw,
            flags=re.IGNORECASE
        )

        if match:

            candidate = (
                strip_html(
                    match.group(1)
                )
            )

            if candidate:

                return candidate


    return None


# ============================================================
# STATION NAME FINDER
# ============================================================

KNOWN_NAME_KEYS = [

    "STN_NAME",
    "STN_NAM_DE",
    "STN_NAM",
    "STATION_NAME",
    "STATION",
    "MRT_NAME",

    "stn_name",
    "station_name",
    "station",

    "NAME",
    "Name",
    "name"
]


def looks_like_station_name(
    value
):

    if value is None:

        return False


    text = strip_html(
        value
    )


    upper = (
        text.upper()
    )


    if len(text) < 2:

        return False


    keywords = [

        "MRT",
        "STATION",
        "LRT"
    ]


    return any(
        keyword in upper
        for keyword in keywords
    )


def find_station_name(
    properties
):

    if not properties:

        return None


    # --------------------------------------------------------
    # 1. Known fields
    # --------------------------------------------------------

    for key in KNOWN_NAME_KEYS:

        if key not in properties:

            continue


        value = properties.get(
            key
        )


        if (
            value is not None
            and
            str(value).strip()
        ):

            return strip_html(
                value
            )


    # --------------------------------------------------------
    # 2. HTML fields / Description
    # --------------------------------------------------------

    for key, value in properties.items():

        if value is None:

            continue


        extracted = (
            extract_station_from_html(
                value
            )
        )


        if extracted:

            return extracted


    # --------------------------------------------------------
    # 3. Look through arbitrary values
    # --------------------------------------------------------

    for key, value in properties.items():

        if not looks_like_station_name(
            value
        ):

            continue


        text = strip_html(
            value
        )


        # Avoid huge descriptions
        if len(text) <= 150:

            return text


    return None


# ============================================================
# CLEAN STATION NAME
# ============================================================

def clean_station_name(
    name
):

    if not name:

        return None


    name = strip_html(
        name
    )


    # --------------------------------------------------------
    # Remove common prefixes/codes
    # --------------------------------------------------------

    name = re.sub(
        r"^[A-Z]{1,3}\d{1,3}\s*[-–—:]?\s*",
        "",
        name,
        flags=re.IGNORECASE
    )


    # --------------------------------------------------------
    # Remove MRT/LRT station suffix
    # --------------------------------------------------------

    name = re.sub(
        r"\bMRT\s+STATION\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\bLRT\s+STATION\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\bMRT\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = re.sub(
        r"\bLRT\b",
        "",
        name,
        flags=re.IGNORECASE
    )


    name = re.sub(
        r"\s+",
        " ",
        name
    )


    name = (
        name
        .strip(
            " -–—,;"
        )
        .strip()
    )


    if not name:

        return None


    # --------------------------------------------------------
    # Title case without destroying apostrophes badly
    # --------------------------------------------------------

    return (
        name.title()
    )


# ============================================================
# GEOMETRY COORDINATE EXTRACTION
# ============================================================

def flatten_coordinate_pairs(
    coordinates
):

    pairs = []


    if coordinates is None:

        return pairs


    # Normal [lon, lat]
    if (
        isinstance(
            coordinates,
            (list, tuple)
        )
        and
        len(coordinates) >= 2
        and
        isinstance(
            coordinates[0],
            (int, float)
        )
        and
        isinstance(
            coordinates[1],
            (int, float)
        )
    ):

        pairs.append(
            (
                float(
                    coordinates[0]
                ),
                float(
                    coordinates[1]
                )
            )
        )

        return pairs


    if isinstance(
        coordinates,
        (list, tuple)
    ):

        for child in coordinates:

            pairs.extend(
                flatten_coordinate_pairs(
                    child
                )
            )


    return pairs


# ============================================================
# EXTRACT MRT EXITS
# ============================================================

station_points = {}

unparsed_features = 0

parsed_exit_count = 0


for feature in features:

    properties = (
        feature.get(
            "properties",
            {}
        )
        or
        {}
    )


    geometry = (
        feature.get(
            "geometry",
            {}
        )
        or
        {}
    )


    station_name = (
        find_station_name(
            properties
        )
    )


    station_name = (
        clean_station_name(
            station_name
        )
    )


    coordinates = (
        flatten_coordinate_pairs(
            geometry.get(
                "coordinates"
            )
        )
    )


    if (
        not station_name
        or
        not coordinates
    ):

        unparsed_features += 1

        continue


    valid_coordinates = []


    for lon, lat in coordinates:

        # Singapore sanity check
        if (
            1.10 <= lat <= 1.55
            and
            103.55 <= lon <= 104.15
        ):

            valid_coordinates.append(
                (
                    lat,
                    lon
                )
            )


    if not valid_coordinates:

        unparsed_features += 1

        continue


    if (
        station_name
        not in
        station_points
    ):

        station_points[
            station_name
        ] = []


    station_points[
        station_name
    ].extend(
        valid_coordinates
    )


    parsed_exit_count += len(
        valid_coordinates
    )


# ============================================================
# CREATE ONE RECORD PER MRT STATION
# ============================================================

MRT_DESTINATIONS = []


for station_name, points in sorted(
    station_points.items()
):

    # --------------------------------------------------------
    # Mean coordinate of station exits
    # --------------------------------------------------------

    mean_lat = float(
        np.mean([
            point[0]
            for point in points
        ])
    )


    mean_lon = float(
        np.mean([
            point[1]
            for point in points
        ])
    )


    station_slug = (
        re.sub(
            r"[^a-z0-9]+",
            "-",
            station_name.lower()
        )
        .strip("-")
    )


    MRT_DESTINATIONS.append({

        "id":
            f"{station_slug}-mrt",

        "name":
            f"{station_name} MRT",

        "station":
            station_name,

        "lat":
            mean_lat,

        "lon":
            mean_lon,

        "exit_count":
            len(points),

        "type":
            "MRT"
    })


# ============================================================
# SAFETY CHECK
# ============================================================

if not MRT_DESTINATIONS:

    print()
    print(
        "========================================"
    )

    print(
        "NO MRT STATIONS PARSED"
    )

    print(
        "========================================"
    )


    # Print first feature so the file structure is visible
    if features:

        print()
        print(
            "First feature properties:"
        )

        print(
            json.dumps(
                features[0].get(
                    "properties",
                    {}
                ),
                indent=2,
                ensure_ascii=False
            )
        )


    raise RuntimeError(
        "No MRT destinations found."
    )


# ============================================================
# SAVE
# ============================================================

safe_json_dump(
    MRT_DESTINATIONS_PATH,
    MRT_DESTINATIONS
)


# ============================================================
# SUMMARY
# ============================================================

print()
print(
    "========================================"
)

print(
    "MRT DESTINATIONS READY"
)

print(
    "========================================"
)

print(
    "GeoJSON features:",
    len(features)
)

print(
    "Parsed exits:",
    parsed_exit_count
)

print(
    "Unparsed features:",
    unparsed_features
)

print(
    "Unique MRT stations:",
    len(
        MRT_DESTINATIONS
    )
)

print()
print(
    "First 15 stations:"
)

for station in (
    MRT_DESTINATIONS[:15]
):

    print(
        "•",
        station["name"],
        "| exits:",
        station["exit_count"],
        "|",
        round(
            station["lat"],
            5
        ),
        ",",
        round(
            station["lon"],
            5
        )
    )

print()
print(
    "Saved:"
)

print(
    MRT_DESTINATIONS_PATH
)

print(
    "========================================"
)

MRT GeoJSON features: 613

MRT DESTINATIONS READY
GeoJSON features: 613
Parsed exits: 596
Unparsed features: 17
Unique MRT stations: 180

First 15 stations:
• Admiralty MRT | exits: 4 | 1.44041 , 103.80086
• Aljunied MRT | exits: 2 | 1.31636 , 103.88248
• Ang Mo Kio MRT | exits: 4 | 1.36962 , 103.84968
• Bakau MRT | exits: 2 | 1.38801 , 103.90546
• Bangkit MRT | exits: 1 | 1.37985 , 103.77259
• Bartley MRT | exits: 2 | 1.34287 , 103.87958
• Bayfront MRT | exits: 5 | 1.28209 , 103.8592
• Bayshore MRT | exits: 5 | 1.31281 , 103.94158
• Beauty World MRT | exits: 3 | 1.34109 , 103.77573
• Bedok MRT | exits: 3 | 1.32397 , 103.92941
• Bedok North MRT | exits: 4 | 1.33452 , 103.91851
• Bedok Reservoir MRT | exits: 2 | 1.33645 , 103.93319
• Bencoolen MRT | exits: 3 | 1.29836 , 103.84989
• Bendemeer MRT | exits: 2 | 1.31367 , 103.86315
• Bishan MRT | exits: 5 | 1.35086 , 103.84869

Saved:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\mrt_destinations.json


In [67]:
# ============================================================
# PAIR EACH RESIDENTIAL ORIGIN WITH NEAREST MRT
# ============================================================

def nearest_mrt(
    cluster
):

    return min(

        MRT_DESTINATIONS,

        key=lambda mrt:
            haversine_m(
                cluster[
                    "lat"
                ],
                cluster[
                    "lon"
                ],
                mrt[
                    "lat"
                ],
                mrt[
                    "lon"
                ]
            )
    )


for cluster in clusters:

    destination = (
        nearest_mrt(
            cluster
        )
    )


    distance_to_mrt = (
        haversine_m(
            cluster[
                "lat"
            ],
            cluster[
                "lon"
            ],
            destination[
                "lat"
            ],
            destination[
                "lon"
            ]
        )
    )


    cluster[
        "destination"
    ] = (
        destination
    )


    cluster[
        "straight_line_to_mrt_m"
    ] = round(
        distance_to_mrt,
        1
    )


safe_json_dump(
    RESIDENTIAL_CLUSTERS_PATH,
    clusters
)


print(
    "Residential clusters paired with MRT destinations."
)

print()


for cluster in (
    clusters[:10]
):

    print(
        cluster[
            "name"
        ],
        "→",
        cluster[
            "destination"
        ][
            "name"
        ],
        "|",
        cluster[
            "straight_line_to_mrt_m"
        ],
        "m"
    )

Residential clusters paired with MRT destinations.

Residential Cluster 001 → Kranji MRT | 371.3 m
Residential Cluster 002 → Springleaf MRT | 1134.0 m
Residential Cluster 003 → Khatib MRT | 1491.1 m
Residential Cluster 004 → Stevens MRT | 1307.9 m
Residential Cluster 005 → Sembawang MRT | 1029.9 m
Residential Cluster 006 → Springleaf MRT | 965.2 m
Residential Cluster 007 → King Albert Park MRT | 1242.5 m
Residential Cluster 008 → Upper Thomson MRT | 1311.6 m
Residential Cluster 009 → Tan Kah Kee MRT | 1388.4 m
Residential Cluster 010 → Pasir Panjang MRT | 1019.7 m


In [68]:
# ============================================================
# CELL 16 — WALKING ROUTING
# ============================================================

def get_walking_route(
    origin,
    destination
):

    cache_key = (
        f"{origin['lat']:.6f},"
        f"{origin['lon']:.6f}"
        "__"
        f"{destination['lat']:.6f},"
        f"{destination['lon']:.6f}"
    )


    cached = (
        cache_get(
            "route_cache",
            cache_key
        )
    )


    if cached is not None:

        return cached


    url = (
        f"{FOOT_ROUTER}/"
        f"{origin['lon']},"
        f"{origin['lat']};"
        f"{destination['lon']},"
        f"{destination['lat']}"
    )


    response = (
        HTTP.get(
            url,

            params={
                "overview":
                    "full",

                "geometries":
                    "geojson",

                "steps":
                    "false"
            },

            timeout=60
        )
    )


    response.raise_for_status()


    payload = (
        response.json()
    )


    if not payload.get(
        "routes"
    ):

        raise RuntimeError(
            "No walking route returned."
        )


    route = (
        payload[
            "routes"
        ][0]
    )


    coordinates = (
        route[
            "geometry"
        ][
            "coordinates"
        ]
    )


    points = [
        {
            "lat":
                float(lat),

            "lon":
                float(lon)
        }

        for lon, lat in
        coordinates
    ]


    result = {
        "points":
            points,

        "distance_m":
            float(
                route.get(
                    "distance",
                    0
                )
            ),

        "duration_s":
            float(
                route.get(
                    "duration",
                    0
                )
            )
    }


    cache_set(
        "route_cache",
        cache_key,
        result
    )


    return result


print(
    "Walking router ready."
)

Walking router ready.


In [69]:
# ============================================================
# CELL 17 — PRELIMINARY ROUTES
# ============================================================

preliminary_routes = {}


route_start_time = (
    time.time()
)


total_clusters = (
    len(
        clusters
    )
)


for index, cluster in enumerate(
    clusters,
    start=1
):

    destination = (
        cluster[
            "destination"
        ]
    )


    route_id = (
        f"{cluster['id']}"
        "__"
        f"{destination['id']}"
    )


    print(
        f"[{index:03d}/{total_clusters}]",
        cluster[
            "name"
        ],
        "→",
        destination[
            "name"
        ]
    )


    try:

        walking_route = (
            get_walking_route(
                cluster,
                destination
            )
        )


        preliminary_routes[
            route_id
        ] = {

            "id":
                route_id,

            "origin":
                {
                    "id":
                        cluster[
                            "id"
                        ],

                    "name":
                        cluster[
                            "name"
                        ],

                    "lat":
                        cluster[
                            "lat"
                        ],

                    "lon":
                        cluster[
                            "lon"
                        ]
                },

            "destination":
                destination,

            "straight_line_distance_m":
                cluster[
                    "straight_line_to_mrt_m"
                ],

            "distance_m":
                walking_route[
                    "distance_m"
                ],

            "duration_s":
                walking_route[
                    "duration_s"
                ],

            "points":
                walking_route[
                    "points"
                ]
        }


    except Exception as error:

        print(
            "    FAILED:",
            str(
                error
            )[:120]
        )


safe_json_dump(
    PRELIMINARY_ROUTE_PATH,

    {
        "routesById":
            preliminary_routes
    }
)


print()

print(
    "=============================================="
)

print(
    "PRELIMINARY ROUTES COMPLETE"
)

print(
    "=============================================="
)

print(
    "Routes generated:",
    len(
        preliminary_routes
    )
)

print(
    "Time:",
    format_seconds(
        time.time()
        -
        route_start_time
    )
)

[001/120] Residential Cluster 001 → Kranji MRT
[002/120] Residential Cluster 002 → Springleaf MRT
[003/120] Residential Cluster 003 → Khatib MRT
[004/120] Residential Cluster 004 → Stevens MRT
[005/120] Residential Cluster 005 → Sembawang MRT
[006/120] Residential Cluster 006 → Springleaf MRT
[007/120] Residential Cluster 007 → King Albert Park MRT
[008/120] Residential Cluster 008 → Upper Thomson MRT
[009/120] Residential Cluster 009 → Tan Kah Kee MRT
[010/120] Residential Cluster 010 → Pasir Panjang MRT
[011/120] Residential Cluster 011 → Jurong East MRT
[012/120] Residential Cluster 012 → Dover MRT
[013/120] Residential Cluster 013 → Harbourfront MRT
[014/120] Residential Cluster 014 → Choa Chu Kang MRT
[015/120] Residential Cluster 015 → Queenstown MRT
[016/120] Residential Cluster 016 → Oasis MRT
[017/120] Residential Cluster 017 → Canberra MRT
[018/120] Residential Cluster 018 → Khatib MRT
[019/120] Residential Cluster 019 → Labrador Park MRT
[020/120] Residential Cluster 020 → W

In [70]:
# ============================================================
# CELL 18 — MAPILLARY TOKEN
# ============================================================

MAPILLARY_TOKEN = (
    os.getenv(
        "MAPILLARY_TOKEN"
    )
)


if not MAPILLARY_TOKEN:

    MAPILLARY_TOKEN = (
        input(
            "Enter Mapillary token: "
        )
        .strip()
    )


if not MAPILLARY_TOKEN:

    raise RuntimeError(
        "Mapillary token required."
    )


print(
    "Mapillary token ready."
)

Enter Mapillary token:  MLY|28290634317266688|f72f1fb60d05bb22ff3c0e8ab253e95b


Mapillary token ready.


In [71]:
# ============================================================
# CELL 19 — MAPILLARY IMAGE CRAWLER
# ============================================================

def route_bbox(
    points,
    padding_deg=0.001
):

    lats = [
        point[
            "lat"
        ]
        for point in
        points
    ]

    lons = [
        point[
            "lon"
        ]
        for point in
        points
    ]


    return (
        min(
            lons
        )
        -
        padding_deg,

        min(
            lats
        )
        -
        padding_deg,

        max(
            lons
        )
        +
        padding_deg,

        max(
            lats
        )
        +
        padding_deg
    )


def fetch_mapillary_candidates(
    route_points
):

    bbox_string = ",".join(
        map(
            str,
            route_bbox(
                route_points
            )
        )
    )


    cache_key = (
        hashlib.sha1(
            bbox_string.encode(
                "utf-8"
            )
        )
        .hexdigest()
    )


    cached = (
        cache_get(
            "mapillary_cache",
            cache_key
        )
    )


    if cached is not None:

        return cached


    response = (
        HTTP.get(
            MAPILLARY_IMAGES_URL,

            params={
                "access_token":
                    MAPILLARY_TOKEN,

                "bbox":
                    bbox_string,

                "limit":
                    2000,

                "fields":
                    (
                        "id,"
                        "thumb_1024_url,"
                        "computed_geometry,"
                        "captured_at"
                    )
            },

            timeout=90
        )
    )


    response.raise_for_status()


    candidates = (
        response
        .json()
        .get(
            "data",
            []
        )
    )


    cache_set(
        "mapillary_cache",
        cache_key,
        candidates
    )


    return candidates


print(
    "Mapillary crawling ready."
)

Mapillary crawling ready.


In [72]:
# ============================================================
# CELL 20 — MATCH IMAGES TO CHECKPOINTS
# ============================================================

def match_images_to_checkpoints(
    checkpoints,
    candidates
):

    usable = []


    for image in candidates:

        geometry = (
            image.get(
                "computed_geometry"
            )
        )


        if (
            not geometry
            or
            geometry.get(
                "type"
            )
            !=
            "Point"
        ):

            continue


        coordinates = (
            geometry.get(
                "coordinates",
                []
            )
        )


        if (
            len(
                coordinates
            )
            <
            2
        ):

            continue


        usable.append({

            **image,

            "lon":
                float(
                    coordinates[0]
                ),

            "lat":
                float(
                    coordinates[1]
                )
        })


    if not usable:

        return [
            []
            for _ in
            checkpoints
        ]


    mean_lat = (
        np.mean([
            image[
                "lat"
            ]
            for image in
            usable
        ])
    )


    lat_scale = (
        111_320
    )


    lon_scale = (
        111_320
        *
        math.cos(
            math.radians(
                mean_lat
            )
        )
    )


    image_xy = (
        np.array([
            [
                image[
                    "lon"
                ]
                *
                lon_scale,

                image[
                    "lat"
                ]
                *
                lat_scale
            ]

            for image in
            usable
        ])
    )


    tree = (
        cKDTree(
            image_xy
        )
    )


    results = []


    for checkpoint in checkpoints:

        checkpoint_xy = [

            checkpoint[
                "lon"
            ]
            *
            lon_scale,

            checkpoint[
                "lat"
            ]
            *
            lat_scale
        ]


        k = min(
            MAPILLARY_CANDIDATES_PER_CHECKPOINT,
            len(
                usable
            )
        )


        distances, indices = (
            tree.query(
                checkpoint_xy,
                k=k
            )
        )


        distances = (
            np.atleast_1d(
                distances
            )
        )


        indices = (
            np.atleast_1d(
                indices
            )
        )


        matches = []


        for distance, index in zip(
            distances,
            indices
        ):

            distance = (
                float(
                    distance
                )
            )


            if (
                distance
                >
                MAPILLARY_MAX_DISTANCE_M
            ):

                continue


            image = (
                usable[
                    int(
                        index
                    )
                ]
                .copy()
            )


            image[
                "distance_m"
            ] = (
                round(
                    distance,
                    2
                )
            )


            matches.append(
                image
            )


        results.append(
            matches
        )


    return results


print(
    "Mapillary checkpoint matcher ready."
)

Mapillary checkpoint matcher ready.


In [73]:
# ============================================================
# CELL 21 — SUPPORT ENVIRONMENT + PCN MATCHING
# ============================================================

def route_buffer_geometry(
    route_points,
    distance_m
):

    point_gdf = (
        gpd.GeoDataFrame(

            geometry=[
                Point(
                    point[
                        "lon"
                    ],
                    point[
                        "lat"
                    ]
                )

                for point in
                route_points
            ],

            crs="EPSG:4326"
        )
        .to_crs(
            epsg=3414
        )
    )


    return (
        point_gdf
        .geometry
        .unary_union
        .buffer(
            distance_m
        )
    )


def environment_ids_near_route(
    route_points
):

    if (
        SUPPORT_ENVIRONMENT_GDF.empty
    ):

        return []


    route_buffer = (
        route_buffer_geometry(
            route_points,
            SUPPORT_ENVIRONMENT_DISTANCE_M
        )
    )


    mask = (
        SUPPORT_ENVIRONMENT_GDF
        .geometry
        .intersects(
            route_buffer
        )
    )


    return (
        SUPPORT_ENVIRONMENT_GDF[
            mask
        ][
            "id"
        ]
        .tolist()
    )


def corridor_ids_near_route(
    route_points
):

    if (
        SUPPORT_CORRIDOR_GDF.empty
    ):

        return []


    route_buffer = (
        route_buffer_geometry(
            route_points,
            30
        )
    )


    mask = (
        SUPPORT_CORRIDOR_GDF
        .geometry
        .intersects(
            route_buffer
        )
    )


    return (
        SUPPORT_CORRIDOR_GDF[
            mask
        ][
            "id"
        ]
        .tolist()
    )


print(
    "Support-environment matching ready."
)

Support-environment matching ready.


In [74]:
# ============================================================
# CELL 22 — BUILD FINAL ROUTE LIBRARY
# ============================================================

routes_by_id = {}


route_items = list(
    preliminary_routes.items()
)


total_routes = (
    len(
        route_items
    )
)


build_start = (
    time.time()
)


for route_number, (
    route_id,
    route
) in enumerate(
    route_items,
    start=1
):

    route_start = (
        time.time()
    )


    print()
    print(
        "=============================================="
    )

    print(
        f"[{route_number:03d}/{total_routes}]",
        route_id
    )


    route_points = (
        route[
            "points"
        ]
    )


    # ========================================================
    # CHECKPOINTS
    # ========================================================

    checkpoints = (
        sample_route(
            route_points,
            ROUTE_CHECKPOINT_SPACING_M
        )
    )


    print(
        "checkpoints:",
        len(
            checkpoints
        )
    )


    # ========================================================
    # POINT FAMILY SUPPORT
    # ========================================================

    support_ids = (
        support_ids_near_route(
            route_points
        )
    )


    print(
        "recorded support nodes:",
        len(
            support_ids
        )
    )


    # ========================================================
    # SUPPORT ENVIRONMENTS
    # ========================================================

    environment_ids = (
        environment_ids_near_route(
            route_points
        )
    )


    print(
        "support environments:",
        len(
            environment_ids
        )
    )


    # ========================================================
    # PARK CONNECTOR
    # ========================================================

    corridor_ids = (
        corridor_ids_near_route(
            route_points
        )
    )


    print(
        "park connector corridors:",
        len(
            corridor_ids
        )
    )


    # ========================================================
    # MAPILLARY
    # ========================================================

    try:

        candidates = (
            fetch_mapillary_candidates(
                route_points
            )
        )


        image_matches = (
            match_images_to_checkpoints(
                checkpoints,
                candidates
            )
        )


        print(
            "Mapillary candidates:",
            len(
                candidates
            )
        )


    except Exception as error:

        print(
            "Mapillary failed:",
            str(
                error
            )[:120]
        )


        image_matches = [
            []
            for _ in
            checkpoints
        ]


    # ========================================================
    # CHECKPOINT RECORDS
    # ========================================================

    checkpoint_records = []


    cumulative_distance = (
        0.0
    )


    previous = None


    for checkpoint_index, (
        checkpoint,
        images
    ) in enumerate(
        zip(
            checkpoints,
            image_matches
        )
    ):


        if (
            previous
            is not None
        ):

            cumulative_distance += (
                haversine_m(
                    previous[
                        "lat"
                    ],
                    previous[
                        "lon"
                    ],
                    checkpoint[
                        "lat"
                    ],
                    checkpoint[
                        "lon"
                    ]
                )
            )


        previous = (
            checkpoint
        )


        # ----------------------------------------------------
        # SUPPORT NODES NEAR THIS CHECKPOINT
        # ----------------------------------------------------

        nearby_support_ids = []


        for support_id in (
            support_ids
        ):

            support = (
                SUPPORT_BY_ID[
                    support_id
                ]
            )


            support_distance = (
                haversine_m(
                    checkpoint[
                        "lat"
                    ],
                    checkpoint[
                        "lon"
                    ],
                    support[
                        "lat"
                    ],
                    support[
                        "lon"
                    ]
                )
            )


            if (
                support_distance
                <=
                SUPPORT_ROUTE_DISTANCE_M
            ):

                nearby_support_ids.append(
                    support_id
                )


        image_records = []


        for image in images:

            image_records.append({

                "id":
                    image.get(
                        "id"
                    ),

                "url":
                    image.get(
                        "thumb_1024_url"
                    ),

                "distance_m":
                    image.get(
                        "distance_m"
                    ),

                "captured_at":
                    image.get(
                        "captured_at"
                    )
            })


        checkpoint_records.append({

            "id":
                (
                    f"{route_id}"
                    f"__cp_"
                    f"{checkpoint_index:03d}"
                ),

            "index":
                checkpoint_index,

            "distance_from_origin_m":
                round(
                    cumulative_distance,
                    2
                ),

            "lat":
                checkpoint[
                    "lat"
                ],

            "lon":
                checkpoint[
                    "lon"
                ],

            # -----------------------------------------------
            # RECORDED SUPPORT
            # -----------------------------------------------

            "recordedSupport":
                bool(
                    nearby_support_ids
                ),

            "recordedSupportIds":
                nearby_support_ids,

            # -----------------------------------------------
            # STREET IMAGE
            # -----------------------------------------------

            "imageCandidates":
                image_records,

            # -----------------------------------------------
            # FILLED BY IMAGE ANALYSIS LATER
            # -----------------------------------------------

            "visualRestSupport":
                None,

            "visualRestSupportConfidence":
                None,

            "environmentalFeatures":
                None,

            "environmentCluster":
                None,

            "pca":
                None,

            "supportNeed":
                None,

            "supportNeedConfidence":
                None,

            "recommendedRestNode":
                None
        })


    # ========================================================
    # ROUTE RECORD
    # ========================================================

    routes_by_id[
        route_id
    ] = {

        **route,

        "recordedSupportIds":
            support_ids,

        "recordedSupport":
            [
                SUPPORT_BY_ID[
                    support_id
                ]

                for support_id in
                support_ids
            ],

        "supportEnvironmentIds":
            environment_ids,

        "mobilityCorridorIds":
            corridor_ids,

        "checkpoints":
            checkpoint_records,

        "recommendedNodes":
            [],

        "analysisReady":
            False
    }


    # ========================================================
    # SAVE AFTER EVERY ROUTE
    # ========================================================

    safe_json_dump(
        ROUTE_PROGRESS_PATH,

        {
            "routesById":
                routes_by_id
        }
    )


    elapsed = (
        time.time()
        -
        build_start
    )


    average = (
        elapsed
        /
        route_number
    )


    remaining = (
        total_routes
        -
        route_number
    )


    eta = (
        average
        *
        remaining
    )


    print(
        "route time:",
        round(
            time.time()
            -
            route_start,
            1
        ),
        "sec"
    )


    print(
        "progress:",
        f"{route_number}/{total_routes}",
        f"({route_number / total_routes * 100:.1f}%)"
    )


    print(
        "ETA:",
        format_seconds(
            eta
        )
    )


[001/120] residential-0001__kranji-mrt
checkpoints: 31
recorded support nodes: 0
support environments: 0
park connector corridors: 0
Mapillary candidates: 1302
route time: 0.1 sec
progress: 1/120 (0.8%)
ETA: 0:00:08

[002/120] residential-0002__springleaf-mrt
checkpoints: 37
recorded support nodes: 0
support environments: 0
park connector corridors: 0
Mapillary candidates: 1834
route time: 0.1 sec
progress: 2/120 (1.7%)
ETA: 0:00:07

[003/120] residential-0003__khatib-mrt
checkpoints: 26
recorded support nodes: 0
support environments: 0
park connector corridors: 1
Mapillary candidates: 1789
route time: 0.1 sec
progress: 3/120 (2.5%)
ETA: 0:00:07

[004/120] residential-0004__stevens-mrt
checkpoints: 55


C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union
C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union
C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union


recorded support nodes: 1
support environments: 0
park connector corridors: 0
Mapillary candidates: 1873
route time: 0.1 sec
progress: 4/120 (3.3%)
ETA: 0:00:09

[005/120] residential-0005__sembawang-mrt
checkpoints: 19
recorded support nodes: 0
support environments: 0
park connector corridors: 2
Mapillary candidates: 540
route time: 0.1 sec
progress: 5/120 (4.2%)
ETA: 0:00:08

[006/120] residential-0006__springleaf-mrt
checkpoints: 12
recorded support nodes: 3
support environments: 0
park connector corridors: 0
Mapillary candidates: 553


C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union
C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union
C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union
C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union


route time: 0.0 sec
progress: 6/120 (5.0%)
ETA: 0:00:08

[007/120] residential-0007__king-albert-park-mrt
checkpoints: 28
recorded support nodes: 0
support environments: 0
park connector corridors: 3
Mapillary candidates: 1915
route time: 0.1 sec
progress: 7/120 (5.8%)
ETA: 0:00:09

[008/120] residential-0008__upper-thomson-mrt
checkpoints: 30
recorded support nodes: 3
support environments: 0
park connector corridors: 0
Mapillary candidates: 1889


...

route time: 0.5 sec
progress: 118/120 (98.3%)
ETA: 0:00:00

[119/120] residential-0119__bukit-gombak-mrt
checkpoints: 19
recorded support nodes: 1
support environments: 0
park connector corridors: 2
Mapillary candidates: 559


C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union


route time: 0.5 sec
progress: 119/120 (99.2%)
ETA: 0:00:00

[120/120] residential-0120__tampines-east-mrt
checkpoints: 15
recorded support nodes: 0
support environments: 0
park connector corridors: 0
Mapillary candidates: 842


C:\Users\MSI\AppData\Local\Temp\ipykernel_10204\1656330357.py:38: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union


route time: 0.5 sec
progress: 120/120 (100.0%)
ETA: 0:00:00


In [75]:
# ============================================================
# DIAGNOSTIC — FAMILY SUPPORT PIPELINE
# DOES NOT MODIFY DATA
# ============================================================

from collections import Counter

print()
print("============================================================")
print("FAMILY SUPPORT DIAGNOSTIC")
print("============================================================")


# ============================================================
# 1. RAW DATASETS
# ============================================================

print()
print("1. RAW SUPPORT DATASETS")
print("------------------------------------------------------------")

print(
    "Datasets loaded:",
    len(
        SUPPORT_DATASETS
    )
)


for dataset_name, gdf in SUPPORT_DATASETS.items():

    print()
    print(
        f"DATASET: {dataset_name}"
    )

    print(
        "Rows:",
        len(gdf)
    )

    print(
        "CRS:",
        gdf.crs
    )

    print(
        "Geometry types:",
        gdf.geometry
        .geom_type
        .value_counts()
        .to_dict()
    )

    print(
        "Bounds:",
        tuple(
            round(
                value,
                5
            )
            for value in
            gdf.total_bounds
        )
    )

    print(
        "Columns:",
        list(
            gdf.columns
        )
    )


# ============================================================
# 2. INSPECT ACTUAL TEXT VALUES
# ============================================================

print()
print()
print("2. POSSIBLE CLASS / NAME VALUES")
print("------------------------------------------------------------")


LIKELY_TEXT_COLUMNS = [
    "CLASS",
    "NAME",
    "Name",
    "name",
    "DESCRIPTION",
    "Description",
    "DESC",
    "FACILITY",
    "TYPE",
    "AMENITY",
    "CATEGORY"
]


for dataset_name, gdf in SUPPORT_DATASETS.items():

    print()
    print(
        f"--- {dataset_name} ---"
    )

    found_column = False

    for column in LIKELY_TEXT_COLUMNS:

        if column not in gdf.columns:
            continue

        found_column = True

        values = (
            gdf[
                column
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .value_counts()
            .head(30)
        )

        print()
        print(
            f"COLUMN: {column}"
        )

        for value, count in values.items():

            print(
                f"  {count:5d}  {value}"
            )

    if not found_column:

        print(
            "No obvious text field found."
        )


# ============================================================
# 3. TEST KEYWORD CLASSIFICATION DIRECTLY
# ============================================================

print()
print()
print("3. KEYWORD CLASSIFICATION RESULTS")
print("------------------------------------------------------------")


for dataset_name, gdf in SUPPORT_DATASETS.items():

    category_counter = Counter()

    matched_rows = 0

    total_rows = len(
        gdf
    )

    samples = []


    for _, row in gdf.iterrows():

        text = (
            row_text(
                row
            )
        )

        categories = (
            classify_support(
                text
            )
        )

        if categories:

            matched_rows += 1

            for category in categories:

                category_counter[
                    category
                ] += 1

            if len(samples) < 10:

                samples.append(
                    (
                        categories,
                        text[:250]
                    )
                )


    print()
    print(
        dataset_name
    )

    print(
        "Rows:",
        total_rows
    )

    print(
        "Rows matching support keywords:",
        matched_rows
    )

    print(
        "Category counts:",
        dict(
            category_counter
        )
    )

    if samples:

        print(
            "Sample matches:"
        )

        for categories, text in samples:

            print(
                " ",
                categories,
                "→",
                text
            )


# ============================================================
# 4. CURRENT GENERATED SUPPORT OBJECTS
# ============================================================

print()
print()
print("4. GENERATED SUPPORT OBJECTS")
print("------------------------------------------------------------")


print(
    "SUPPORT_NODES:",
    len(
        SUPPORT_NODES
    )
)

print(
    "SUPPORT_ENVIRONMENTS:",
    len(
        SUPPORT_ENVIRONMENTS
    )
)

print(
    "SUPPORT_CORRIDORS:",
    len(
        SUPPORT_CORRIDORS
    )
)


if SUPPORT_NODES:

    print()
    print(
        "First 10 support nodes:"
    )

    for node in SUPPORT_NODES[:10]:

        print(
            node[
                "id"
            ],
            "|",
            node[
                "primaryCategory"
            ],
            "|",
            node[
                "name"
            ],
            "|",
            round(
                node[
                    "lat"
                ],
                5
            ),
            round(
                node[
                    "lon"
                ],
                5
            )
        )


# ============================================================
# 5. CHECK SUPPORT COORDINATES AGAINST SINGAPORE
# ============================================================

print()
print()
print("5. SUPPORT LOCATION SANITY CHECK")
print("------------------------------------------------------------")


if SUPPORT_NODES:

    support_lats = [
        node[
            "lat"
        ]
        for node in
        SUPPORT_NODES
    ]

    support_lons = [
        node[
            "lon"
        ]
        for node in
        SUPPORT_NODES
    ]


    print(
        "Support latitude range:",
        min(
            support_lats
        ),
        "→",
        max(
            support_lats
        )
    )

    print(
        "Support longitude range:",
        min(
            support_lons
        ),
        "→",
        max(
            support_lons
        )
    )


    singapore_like = [

        node
        for node in
        SUPPORT_NODES

        if (
            1.10
            <=
            node[
                "lat"
            ]
            <=
            1.55

            and

            103.55
            <=
            node[
                "lon"
            ]
            <=
            104.15
        )
    ]


    print(
        "Support nodes inside Singapore bounds:",
        len(
            singapore_like
        ),
        "/",
        len(
            SUPPORT_NODES
        )
    )


else:

    print(
        "No point support nodes exist to test."
    )


# ============================================================
# 6. ROUTE LOCATION SANITY CHECK
# ============================================================

print()
print()
print("6. ROUTE LOCATION SANITY CHECK")
print("------------------------------------------------------------")


all_route_points = []


for route in preliminary_routes.values():

    all_route_points.extend(
        route.get(
            "points",
            []
        )
    )


if all_route_points:

    route_lats = [
        point[
            "lat"
        ]
        for point in
        all_route_points
    ]

    route_lons = [
        point[
            "lon"
        ]
        for point in
        all_route_points
    ]


    print(
        "Route latitude range:",
        min(
            route_lats
        ),
        "→",
        max(
            route_lats
        )
    )

    print(
        "Route longitude range:",
        min(
            route_lons
        ),
        "→",
        max(
            route_lons
        )
    )


# ============================================================
# 7. FIND ACTUAL NEAREST SUPPORT DISTANCES
# ============================================================

print()
print()
print("7. NEAREST SUPPORT DISTANCES")
print("------------------------------------------------------------")


if SUPPORT_NODES and SUPPORT_TREE is not None:

    nearest_distances = []


    route_sample = list(
        preliminary_routes.items()
    )[:20]


    for route_id, route in route_sample:

        route_points = (
            sample_route(
                route[
                    "points"
                ],
                100
            )
        )


        route_min_distance = (
            float(
                "inf"
            )
        )


        for point in route_points:

            x = (
                point[
                    "lon"
                ]
                *
                SUPPORT_LON_SCALE
            )

            y = (
                point[
                    "lat"
                ]
                *
                SUPPORT_LAT_SCALE
            )


            distance, index = (
                SUPPORT_TREE.query(
                    [
                        x,
                        y
                    ],
                    k=1
                )
            )


            route_min_distance = min(
                route_min_distance,
                float(
                    distance
                )
            )


        nearest_distances.append(
            (
                route_id,
                route_min_distance
            )
        )


    nearest_distances.sort(
        key=lambda item:
            item[1]
    )


    for route_id, distance in nearest_distances:

        print(
            f"{distance:8.1f} m  {route_id}"
        )


    print()

    print(
        "Closest support point among these routes:",
        round(
            nearest_distances[0][1],
            1
        ),
        "m"
    )


else:

    print(
        "Cannot calculate nearest support distance "
        "because SUPPORT_NODES is empty."
    )


# ============================================================
# 8. TEST CURRENT 80 m THRESHOLD
# ============================================================

print()
print()
print("8. CURRENT ROUTE MATCH THRESHOLD")
print("------------------------------------------------------------")


print(
    "SUPPORT_ROUTE_DISTANCE_M:",
    SUPPORT_ROUTE_DISTANCE_M,
    "m"
)


if SUPPORT_NODES:

    test_thresholds = [
        50,
        80,
        100,
        150,
        200,
        300
    ]


    for threshold in test_thresholds:

        routes_with_match = 0

        total_matches = 0


        for route in preliminary_routes.values():

            matched_ids = (
                support_ids_near_route(
                    route[
                        "points"
                    ],
                    max_distance_m=
                        threshold
                )
            )


            if matched_ids:

                routes_with_match += 1

                total_matches += len(
                    matched_ids
                )


        print(
            f"{threshold:3d} m → "
            f"{routes_with_match:3d} / "
            f"{len(preliminary_routes)} routes "
            f"have support | "
            f"{total_matches} total matches"
        )


# ============================================================
# 9. CURRENT FINAL ROUTE RESULTS
# ============================================================

print()
print()
print("9. FINAL ROUTE RESULT CHECK")
print("------------------------------------------------------------")


if (
    "routes_by_id"
    in globals()
    and
    routes_by_id
):

    support_counts = [
        len(
            route.get(
                "recordedSupportIds",
                []
            )
        )
        for route in
        routes_by_id.values()
    ]


    print(
        "Routes analysed:",
        len(
            routes_by_id
        )
    )

    print(
        "Routes with at least one support node:",
        sum(
            count > 0
            for count in
            support_counts
        )
    )

    print(
        "Maximum support nodes on one route:",
        max(
            support_counts
        )
        if support_counts
        else
        0
    )


else:

    print(
        "routes_by_id not available yet."
    )


print()
print(
    "============================================================"
)

print(
    "DIAGNOSTIC COMPLETE"
)

print(
    "============================================================"
)


FAMILY SUPPORT DIAGNOSTIC

1. RAW SUPPORT DATASETS
------------------------------------------------------------
Datasets loaded: 3

DATASET: CentralNatureReserveAmenities.geojson
Rows: 449
CRS: EPSG:4326
Geometry types: {'Point': 449}
Bounds: (np.float64(103.75332), np.float64(1.2624), np.float64(103.83662), np.float64(1.40562))
Columns: ['OBJECTID', 'BUILD_TYPE', 'BUILD_NAME', 'INC_CRC', 'FMEL_UPD_D', 'geometry']

DATASET: ParkConnectorLoop.geojson
Rows: 878
CRS: EPSG:4326
Geometry types: {'LineString': 784, 'MultiLineString': 94}
Bounds: (np.float64(103.6932), np.float64(1.27294), np.float64(104.03165), np.float64(1.46288))
Columns: ['OBJECTID', 'PARK', 'PCN_LOOP', 'MORE_INFO', 'INC_CRC', 'FMEL_UPD_D', 'SHAPE.LEN', 'geometry']

DATASET: ParkFacilities.geojson
Rows: 14754
CRS: EPSG:4326
Geometry types: {'Point': 14754}
Bounds: (np.float64(103.6897), np.float64(1.21364), np.float64(104.01666), np.float64(1.46318))
Columns: ['OBJECTID', 'UNIQUEID', 'NAME', 'CLASS', 'ADDITIONAL_INFO', '

In [76]:
# ============================================================
# CELL 23 — SAVE FINAL ROUTE LIBRARY
# ============================================================

ROUTE_LIBRARY = {

    "version":
        8,

    "method":
        (
            "Residential origin → nearest MRT → "
            "walking route → family-support opportunities → "
            "support environments → Mapillary imagery"
        ),

    "checkpointSpacingM":
        ROUTE_CHECKPOINT_SPACING_M,

    "supportSearchDistanceM":
        SUPPORT_ROUTE_DISTANCE_M,

    "supportEnvironmentDistanceM":
        SUPPORT_ENVIRONMENT_DISTANCE_M,

    "familySupportLibrary":
        "./family_support_library.json",

    "supportDefinition":
        (
            "Recorded locations or environments where a caregiver "
            "travelling with a young child may reasonably pause, "
            "obtain care, seek shelter or recover during a walking "
            "journey. Absence of recorded support does not imply "
            "that no physical rest opportunity exists."
        ),

    "routesById":
        routes_by_id,

    "routes":
        list(
            routes_by_id.values()
        )
}


safe_json_dump(
    ROUTE_LIBRARY_PATH,
    ROUTE_LIBRARY
)


print()
print(
    "=============================================="
)

print(
    "ROUTE IMAGERY BACKEND COMPLETE"
)

print(
    "=============================================="
)

print(
    "Routes:",
    len(
        routes_by_id
    )
)

print(
    "Saved:"
)

print(
    ROUTE_LIBRARY_PATH
)

print(
    "=============================================="
)


ROUTE IMAGERY BACKEND COMPLETE
Routes: 120
Saved:
C:\Users\MSI\OneDrive\Desktop\family-toilet-search\data\generated\residential_route_library.json


In [77]:
# ============================================================
# CELL 24 — FINAL VERIFICATION
# ============================================================

total_checkpoints = sum(
    len(
        route.get(
            "checkpoints",
            []
        )
    )
    for route in
    routes_by_id.values()
)


total_image_references = sum(
    len(
        checkpoint.get(
            "imageCandidates",
            []
        )
    )
    for route in
    routes_by_id.values()
    for checkpoint in
    route.get(
        "checkpoints",
        []
    )
)


routes_with_support = sum(
    1
    for route in
    routes_by_id.values()
    if route.get(
        "recordedSupportIds"
    )
)


routes_with_environment = sum(
    1
    for route in
    routes_by_id.values()
    if route.get(
        "supportEnvironmentIds"
    )
)


routes_with_pcn = sum(
    1
    for route in
    routes_by_id.values()
    if route.get(
        "mobilityCorridorIds"
    )
)


print(
    "=============================================="
)

print(
    "BACKEND SUMMARY"
)

print(
    "=============================================="
)

print(
    "Residential routes:",
    len(
        routes_by_id
    )
)

print(
    "Checkpoints:",
    total_checkpoints
)

print(
    "Mapillary image references:",
    total_image_references
)

print(
    "Routes with recorded family support:",
    routes_with_support
)

print(
    "Routes intersecting supportive environments:",
    routes_with_environment
)

print(
    "Routes near park connectors:",
    routes_with_pcn
)

print(
    "=============================================="
)

BACKEND SUMMARY
Residential routes: 120
Checkpoints: 2685
Mapillary image references: 3962
Routes with recorded family support: 26
Routes intersecting supportive environments: 0
Routes near park connectors: 57
